In [ ]:
# Install required packages
pip install python-jose cryptography 2>&1 | tail -5 install -q langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# A5 – Re-ranking

- **Adapted from:** `all_rag_techniques/reranking.ipynb`
- **Experiment ID:** `A5_RERANK`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Compare retrieval with and without re-ranking. Takes Top-30 candidates from hybrid retrieval (A4), then applies LLM-based reranker to select Top-5.

## Hypothesis

Re-ranking should improve Precision@K and MRR by pushing truly relevant documents higher, filtering out hard negatives that passed the initial retrieval stage.

## Control Variables

- Chunking: best from A1/A2
- Query: best from A3 (or original)
- Retrieval: hybrid RRF from A4 (candidate Top-30)
- Prompt: same naive prompt
- LLM: same

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A5_RERANK"
NOTEBOOK = "06_reranking.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]
FINAL_TOP_K = config["retrieval"]["final_top_k"]
CANDIDATE_TOP_K = config["retrieval"]["candidate_top_k"]
RRF_K = config["retrieval"]["rrf_k"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f"LLM OK: {llm.invoke('hi').content[:30]}")
print(f"Embedding OK: dim={len(embeddings.embed_query('test'))}")

## 2. Build Indexes (same as A4)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)
for c in chunks:
    c.page_content = c.page_content.replace('\t', ' ')

# Dense index
vectorstore = FAISS.from_documents(chunks, embeddings)

# BM25 index
chunk_texts = [c.page_content for c in chunks]
tokenized_chunks = [text.lower().split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

print(f"Indexes built: {len(chunks)} chunks")

# Load questions
with open(PROJECT_ROOT / config["paths"]["questions"], "r", encoding="utf-8") as f:
    eval_questions = json.load(f)
print(f"Questions: {len(eval_questions)}")

## 3. Hybrid Retrieval (from A4)

In [ ]:
def retrieve_hybrid_rrf(query: str, k: int = CANDIDATE_TOP_K):
    """Hybrid RRF retrieval returning top-k candidates."""
    # Dense
    dense_docs = vectorstore.similarity_search(query, k=k)
    # BM25
    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top = np.argsort(bm25_scores)[::-1][:k]

    # RRF fusion
    rrf_scores = {}
    for rank, doc in enumerate(dense_docs, 1):
        key = id(doc)
        rrf_scores[key] = {"score": 1.0 / (RRF_K + rank), "doc": doc}

    for rank, idx in enumerate(bm25_top, 1):
        doc = chunks[idx]
        key = id(doc)
        if key in rrf_scores:
            rrf_scores[key]["score"] += 1.0 / (RRF_K + rank)
        else:
            rrf_scores[key] = {"score": 1.0 / (RRF_K + rank), "doc": doc}

    sorted_results = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [(item["doc"], item["score"]) for item in sorted_results[:k]]

## 4. LLM-based Reranker

Scores each (query, document) pair on a 1-10 scale using the LLM.

In [ ]:
RERANK_PROMPT = PromptTemplate(
    input_variables=["query", "document"],
    template="""Rate the relevance of this document to the query on a scale of 1-10.
Respond with ONLY a number.

Query: {query}
Document: {document}
Score:"""
)
rerank_chain = RERANK_PROMPT | llm


def rerank_documents(query: str, candidates: list, top_k: int = FINAL_TOP_K):
    """Rerank candidates using LLM scoring."""
    scored = []
    for doc, initial_score in candidates:
        try:
            resp = rerank_chain.invoke({"query": query, "document": doc.page_content[:500]})
            score_text = resp.content.strip()
            score = float(''.join(c for c in score_text if c.isdigit() or c == '.')[:4])
        except (ValueError, IndexError):
            score = 5.0
        scored.append((doc, score, initial_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

print(f"Reranker: LLM-based, candidates={CANDIDATE_TOP_K} -> final={FINAL_TOP_K}")

## 5. Run Evaluation (No Rerank vs Rerank)

In [ ]:
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)
answer_chain = NAIVE_PROMPT | llm

all_results = []

for q in eval_questions:
    question = q["question"]
    relevant_docs = q.get("relevant_documents", [])

    # --- Hybrid without rerank (take top FINAL_TOP_K directly) ---
    with Timer() as t_ret_no:
        candidates = retrieve_hybrid_rrf(question, k=CANDIDATE_TOP_K)
    no_rerank_docs = [doc for doc, _ in candidates[:FINAL_TOP_K]]
    no_rerank_ids = [d.metadata.get("source", f"c{i}") for i, d in enumerate(no_rerank_docs)]
    context_no = "\n\n".join([d.page_content for d in no_rerank_docs])

    with Timer() as t_gen_no:
        resp_no = answer_chain.invoke({"context": context_no, "question": question})

    metrics_no = compute_retrieval_metrics(no_rerank_ids, relevant_docs, k=FINAL_TOP_K)
    all_results.append(build_result_record(
        experiment_id=f"{EXPERIMENT_ID}_NO_RERANK",
        notebook=NOTEBOOK, config_hash=CONFIG_HASH, seed=SEED,
        question_id=q["question_id"], question=question,
        answer=resp_no.content, metrics=metrics_no,
        latency={"retrieval_seconds": t_ret_no.elapsed, "rerank_seconds": 0,
                 "generation_seconds": t_gen_no.elapsed,
                 "total_seconds": t_ret_no.elapsed + t_gen_no.elapsed},
        usage={"context_chars": len(context_no), "llm_calls": 1},
        reranked=False,
    ))

    # --- Hybrid with rerank ---
    with Timer() as t_rerank:
        reranked = rerank_documents(question, candidates, top_k=FINAL_TOP_K)
    rerank_docs = [doc for doc, _, _ in reranked]
    rerank_ids = [d.metadata.get("source", f"c{i}") for i, d in enumerate(rerank_docs)]
    context_re = "\n\n".join([d.page_content for d in rerank_docs])

    with Timer() as t_gen_re:
        resp_re = answer_chain.invoke({"context": context_re, "question": question})

    metrics_re = compute_retrieval_metrics(rerank_ids, relevant_docs, k=FINAL_TOP_K)
    all_results.append(build_result_record(
        experiment_id=f"{EXPERIMENT_ID}_RERANKED",
        notebook=NOTEBOOK, config_hash=CONFIG_HASH, seed=SEED,
        question_id=q["question_id"], question=question,
        answer=resp_re.content, metrics=metrics_re,
        latency={"retrieval_seconds": t_ret_no.elapsed, "rerank_seconds": t_rerank.elapsed,
                 "generation_seconds": t_gen_re.elapsed,
                 "total_seconds": t_ret_no.elapsed + t_rerank.elapsed + t_gen_re.elapsed},
        usage={"context_chars": len(context_re), "llm_calls": 1 + CANDIDATE_TOP_K},
        reranked=True,
    ))

    print(f"  [{q['question_id']}] rerank_time={t_rerank.elapsed:.2f}s")

print(f"\nTotal: {len(all_results)} records")

## 6. Comparison Table

In [ ]:
summary_rows = []
for label, reranked_flag in [("NO_RERANK", False), ("RERANKED", True)]:
    subset = [r for r in all_results if r.get("reranked") == reranked_flag]
    metrics_agg = {}
    for key in subset[0]["metrics"]:
        vals = [r["metrics"][key] for r in subset if r["metrics"].get(key) is not None]
        metrics_agg[key] = round(np.mean(vals), 3) if vals else None

    avg_total = np.mean([r["latency"]["total_seconds"] for r in subset])
    avg_rerank = np.mean([r["latency"]["rerank_seconds"] for r in subset])
    avg_llm = np.mean([r["usage"]["llm_calls"] for r in subset])

    summary_rows.append({
        "config": label,
        "avg_rerank_s": round(avg_rerank, 3),
        "avg_total_s": round(avg_total, 3),
        "avg_llm_calls": round(avg_llm, 1),
        **metrics_agg,
    })

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

## 7. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]
save_jsonl(all_results, output_dir / "A5_reranking.jsonl")
save_csv_summary(summary_rows, output_dir / "A5_reranking_summary.csv")
save_config_snapshot(config, output_dir)

## 8. Observations

- Without rerank: Precision@5=`___`, MRR=`___`
- With rerank: Precision@5=`___`, MRR=`___`
- Reranking latency: avg `___`s per query (due to `___` LLM calls)
- Cases where reranker pushed correct doc higher: `___`
- Cases where reranker pushed correct doc lower: `___`
- Trade-off: improved precision vs `___`x latency increase.

_Fill in after running._